> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

# 06. Python 채팅과 대화 이력 — 60분

이 자료는 AI를 처음 배우는 비전공 학부생을 기준으로 작성했습니다. 수식과 용어가 한 번에 이해되지 않아도 괜찮습니다. 먼저 **입력 → 계산 → 출력 → 결과 해석** 의 흐름을 따라 실행하고, 수업 뒤 상세 설명을 다시 읽으세요.

본인 실습 컨테이너의 Python 커널에서 환경 셀부터 실행합니다. 설명 → 예제 → 파일 TODO → 학습자 검증 → 참고 정답 순서입니다.

실행 순서와 저장 위치는 [04.LLM README](README.md), 낯선 용어는 [04.LLM 용어 찾아보기](GLOSSARY.md)에서 바로 확인할 수 있습니다.

- **TODO는 JupyterLab 파일 편집기에서 `exercises/*.py`를 수정하고 저장합니다.** 노트북이 소스를 덮어쓰지 않습니다.
- 미완성 검증이 `AssertionError`로 실패하는 것은 정상입니다. 실패 메시지를 읽고 TODO를 고친 뒤 검증 셀을 다시 실행하세요.
- `load_op`는 저장된 파일을 다시 읽으므로 보통 커널을 재시작할 필요가 없습니다. 단, 일반 `import`로 읽은 모듈은 재시작 또는 reload가 필요할 수 있습니다.
- 완료한 앞 교시 TODO를 다음 훈련에서도 사용합니다. 참고 정답은 별도 `solutions/` 경로에 있으며, 강사가 안내하는 시점에 비교하고 자신의 코드로 재검증합니다.
- 표기 `[B,T,H]`는 텐서의 **shape** 입니다. `B`는 한 번에 처리하는 샘플 수(batch), `T`는 순서 길이(time/sequence), `H`는 내부 표현 크기(hidden), `V`는 token 종류 수(vocabulary)입니다.
- 컨테이너를 재생성하면 패키지가 포함된 최신 이미지가 필요하며, 소스와 `Data`는 호스트에 보존됩니다.

오류가 나면 가장 먼저 현재 노트북의 앞 셀을 순서대로 실행했는지, TODO 파일을 저장했는지, 오류가 난 텐서의 `shape`, `dtype`, `device`가 무엇인지 확인하세요.

<!-- standalone-review:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** Python 3 커널에서 첫 환경 셀의 `/opt/course/bin/python` 경로를 확인합니다. 코드 셀은 Shift+Enter로 실행하고 `[*]`가 끝난 뒤 다음 셀로 이동합니다. TODO 파일을 저장한 뒤 검증 셀을 실행합니다.

### 공용 Ollama와 이 노트북 준비

열 파일은 `/workspace/04.LLM/06_ollama_python.ipynb`입니다. [05_ollama_basics.ipynb의 「Ollama의 client-server 구조」](05_ollama_basics.ipynb)에 기본 구조가 있으며 현재 필요한 내용은 다음과 같습니다. 개인 컨테이너의 Python/CLI가 `http://127.0.0.1:11434`의 공용 서버에 요청하고, LLM 수업 모델은 `gemma3:1b`입니다. 같은 Jetson의 컨테이너가 host network를 쓰므로 같은 주소로 접속합니다. 작은 모델의 loss·훈련 결과·checkpoint는 필요하지 않습니다.

**공용 Ollama 확인:** 첫 환경 셀에서 API와 `gemma3:1b` 모델 목록을 확인합니다. 연결되지 않거나 모델이 없으면 로컬 Setup 담당자에게 공용 서버·모델 저장소 확인을 요청합니다. 다른 수강생은 별도 서버를 만들거나 모델을 다운로드하지 않습니다. Python 표준 라이브러리로 HTTP를 호출하므로 이 노트북 자체는 CUDA 훈련용 import를 요구하지 않습니다.

입력 질문은 현재 셀에 포함되어 있고 API의 `messages`는 `role`·`content` 딕셔너리의 목록입니다. `stream=False`는 완료된 응답 하나를 받으며 생성 내용은 `result['message']['content']`에서 읽습니다. 첫 환경 셀부터 실행해 현재 커널의 `client`와 도우미를 다시 만들고, 해당 절에서 필요한 질문·이력 셀을 순서대로 실행합니다. 저장 로그는 `/workspace/Data/outputs/04.LLM/`에 남고 재부팅 후 메모리 대화 이력은 자동 복원되지 않습니다.

<!-- standalone-review:end -->


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

서버가 앞 대화를 자동으로 기억한다고 생각하면 요청 사이의 차이를 설명하기 어렵습니다. 이 실습은 클라이언트가 성공한 대화의 messages를 저장하고 다음 요청에 다시 보내는 과정을 직접 구현합니다.

### 전체 과정에서 내 코드가 하는 일

```text
기존 이력 복사
  ↓
새 user 메시지 추가
  ↓
서버 요청
  ↓
성공 응답 확정
  ↓
턴 제한·reset
  ↓
실패 시 이력 보존
```

메시지 순서·깊은 복사·이력 제한 TODO를 작성합니다. 실패한 요청이 이력에 절반만 남거나 reset 뒤 옛 대화가 다시 전송되지 않게 검사합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 실패한 요청의 user만 이력에 남겨도 되나요?

**설명:** 현재 계약에서는 성공한 user/assistant 쌍만 확정합니다. 중간 실패는 원래 이력을 보존해야 다음 요청의 문맥이 오염되지 않습니다. fake client 검증과 실제 서버 응답 비교도 별도 근거입니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **1–5절의 수업 핵심 경로** → [2일차 슬라이드 45쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s040) · [배포 PDF 45쪽](../슬라이드/2일차.pdf#page=45)에서 51쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 작은 모델의 훈련 전후 예측 또는 Ollama 응답을 저장합니다.
- **단원 마지막 연결 질문:** 직접 훈련한 숫자 과제와 이미 학습된 모델 사용을 구분합니다. 제공한 작업 지침에 없는 질문에는 근거가 없다고 말하는지 확인합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 시간·목표

이력 없는 요청 10분 → messages TODO 15분 → 이력 있는 요청 15분 → reset·실패·길이 관리 15분 → 정리 5분.

### 수업 진행 경로

1. Ollama 전용 환경 셀과 1절에서 API 요청이 기본적으로 독립적임을 확인합니다.
2. 이력 없는 실제 요청을 실행하고 messages TODO·검증을 완료합니다.
3. 이전 메시지를 포함한 요청을 실행한 뒤 3절 Conversation의 limit/reset을 확인합니다.
4. 4절 단위 테스트와 5절 터미널 채팅을 실행합니다.
5. 2절의 context 비용, deep copy 상세 설명과 마지막 오류 표는 수업 후 복습합니다.

각 셀의 **구분** 표시를 확인하세요. `수업 후 복습·자습` 셀은 건너뛰어도 다음 수업 핵심 셀을 실행할 수 있습니다.

### 앞선 options 실습의 위치와 여기서 쓰는 내용

[05_ollama_basics.ipynb의 「수업 핵심 TODO — options」](05_ollama_basics.ipynb)는 `exercises/options.py`에서 `temperature=0`, `num_ctx=2048`, 양의 정수 `num_predict`의 딕셔너리를 만듭니다. 여기서도 `num_ctx`는 입력과 생성에 쓰는 문맥 용량, `num_predict`는 생성 token 상한이며 글자 수와 다릅니다. 이번 노트북은 `OllamaClient.chat(..., num_predict=32 또는 64)`의 제공 기본값을 직접 사용하므로 **05의 options 변수나 완성 파일 없이 시작할 수 있습니다.** 현재 메시지 TODO만 수정하고, 이력 없는 첫 요청→이력 포함 요청→reset→가짜 응답 검증→로그 저장 순서로 실행하세요.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import json, subprocess, sys, time
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '04.LLM/common').is_dir())
COURSE = ROOT / '04.LLM'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
from common.paths import DATA, OUTPUTS
from common.loading import load_op
from common.ollama_client import OllamaClient, Conversation
OUTPUTS.mkdir(parents=True, exist_ok=True)
probe = OllamaClient()
available = [m['name'] for m in probe.models().get('models', [])]
print('Python:', sys.executable, 'Ollama:', probe.base_url, 'models:', available)
assert 'gemma3:1b' in available, 'Ollama 연결과 gemma3:1b 사전 배포를 확인하세요.'


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 1. API 요청은 기본적으로 독립적입니다

사용자는 채팅 창이 계속 이어지므로 모델이 내부에 대화를 영구 저장한다고 생각하기 쉽습니다. 하지만 일반적인 Chat API에서는 client가 요청마다 필요한 이전 메시지를 `messages` 목록에 다시 넣어 보냅니다. 서버는 그 목록을 현재 요청의 문맥으로 처리합니다.

```text
이력 없는 두 요청
요청 1: [user: "설비 코드는 NEPES-42"]
요청 2: [user: "앞서 말한 코드는?"]       ← 앞 정보가 없음

이력 있는 두 번째 요청
요청 2: [user: "설비 코드는 NEPES-42",
         assistant: "확인했습니다",
         user: "앞서 말한 코드는?"]       ← 앞 정보가 포함됨
```

서버 프로세스가 살아 있는 것, 모델 가중치가 메모리에 적재된 것, 대화 메시지가 유지되는 것은 서로 다른 상태입니다. 모델 적재는 다음 요청을 빠르게 할 수 있지만, 이전 대화 내용을 자동으로 넣어 주지는 않습니다.


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다. 수업 중에는 시간이 남을 때 읽습니다.

## 2. 대화 이력도 모델 입력입니다

<!-- procedure:lesson-16-3ab131:start -->
![성공한 대화만 이력에 확정하기](../교재그림/절차/lesson-16.svg)

성공과 실패는 분기입니다. 실패한 요청의 질문만 이력에 남아 다음 대화에 섞이지 않게 확인하세요.
<!-- procedure:lesson-16-3ab131:end -->


이전 user/assistant 메시지는 모델이 참고할 **추가 token 입력** 입니다. 이력이 길수록 무조건 답이 좋아지는 것은 아닙니다.

- 관련 정보는 후속 질문의 의미를 분명하게 할 수 있습니다.
- 오래되거나 서로 모순되는 정보는 답을 방해할 수 있습니다.
- 입력 token이 늘면 처리 시간과 메모리 사용량이 늘 수 있습니다.
- 모델과 설정의 context 한도를 넘으면 잘림 또는 오류 정책을 고려해야 합니다.
- 비밀번호, 개인정보, 민감한 생산 데이터는 필요 없이 이력에 넣거나 로그로 저장하지 않습니다.

대화 이력은 애플리케이션의 데이터이므로 어떤 사용자·세션에 속하는지 분리해야 합니다. 여러 원격 사용자가 한 Ollama 서버를 공유하더라도 각자의 Python client가 별도 messages를 관리할 수 있습니다.

### 대화 이력이 늘면 답변에 남길 문맥 공간도 고려합니다

여기서 **추가 token 입력** 은 새로운 vocabulary 항목을 등록한다는 의미가 아닙니다. 같은 모델·tokenizer의 token 종류 수는 그대로이고, 이전 대화를 다시 보내면서 입력 token 수가 늘어납니다.

- **`num_ctx`(context 길이)** 는 생성 중 참고하는 문맥의 token 용량이고, **`num_predict`(출력 상한)** 는 새로 생성할 token 수의 최대값입니다.
- 입력 token에는 system 지시, 이전 user/assistant 메시지, 현재 질문과 채팅 템플릿의 구분 token이 포함됩니다.
- 예를 들어 `num_ctx=2048`, `num_predict=256`이면 전체 입력을 유지하고 답변 공간을 확보하도록 **입력 1,792 token 이내 + 생성 여유 256 token** 으로 계획합니다. 종료 token이 먼저 나오면 답변은 256 token보다 짧을 수 있습니다.
- 이 계산은 전체 문맥을 보존하기 위한 예산입니다. 한도 초과 시 잘림·문맥 이동·오류 처리는 모델과 실행 환경에 따라 달라지므로, 실제 출력이 항상 남은 문맥 길이에서 멈춘다는 뜻은 아닙니다.

**진행 힌트:** 현재 질문에 필요한 이력을 고르고 → 답변용 token 여유를 남기고 → 입력이 너무 길면 오래된 대화를 제거하거나 요약합니다. `max_turns`는 대화 쌍 수를 제한하므로 token 길이 제한을 대신하지 못합니다. 필요하면 모델 지원 범위와 메모리 여유 안에서 `num_ctx`를 늘립니다. `num_predict`를 늘리는 것만으로 문맥 용량은 늘어나지 않습니다.

공식 참고: [Ollama 생성 파라미터](https://docs.ollama.com/modelfile#valid-parameters-and-values), [Context 길이와 메모리](https://docs.ollama.com/context-length)



In [ ]:
# 구분: 수업 핵심 셀
client=OllamaClient()
first=client.chat([{'role':'user','content':'이 대화에서 설비 코드는 NEPES-42입니다. 확인했다고만 답해줘.'}],num_predict=32)
without_history=client.chat([{'role':'user','content':'앞서 말한 설비 코드는 무엇인가요?'}],num_predict=64)
print('첫 답:',first['message']['content'])
print('이력 없는 두 번째 답:',without_history['message']['content'])

> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다. 수업 중에는 시간이 남을 때 읽습니다.

### 결과를 해석할 때 주의

이력 없는 두 번째 답이 우연히 `NEPES-42`를 말하지 못해야만 실습이 성공하는 것은 아닙니다. 모델이 예시에서 자주 보이는 문자열을 우연히 추측할 수도 있습니다. 핵심 증거는 자연어 답의 내용 하나가 아니라 **두 번째 요청 payload에 첫 메시지가 없다는 사실** 입니다. 모델의 행동과 client 코드의 상태 관리를 분리해서 검증합니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 수업 핵심 TODO — messages

수정 파일: **[04.LLM/exercises/messages.py](exercises/messages.py)**

previous가 None이면 빈 이력에서 시작하고, 있으면 이전 메시지를 복사합니다. 마지막에 role=user, content=question을 추가하세요.
호출자의 이전 이력과 내부 dict를 변경하지 않도록 깊은 복사를 사용합니다. 반환값은 실제 다음 API 요청에 사용합니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `build_messages(question, previous=None)`는 질문 문자열과 기존 메시지 dict들의 목록(또는 None)을 받아 새 목록을 반환합니다. None과 `'안녕'`이면 `[{'role':'user','content':'안녕'}]`입니다. 기존 메시지가 2개면 반환 목록은 기존 순서의 2개+새 user 메시지 1개입니다.
**작성 흐름 힌트:** None이면 빈 목록, 아니면 deepcopy → 새 dict append → 목록 return. 반환 목록 내부의 예전 dict를 고쳐도 previous의 dict가 변하지 않아야 합니다. 이번 함수에는 별도 질문 검증 계약을 추가하지 않습니다.
**설명할 지점:** “서버가 알아서 대화를 기억하는 것이 아니라 client가 이 목록을 매번 보내는 것입니다. 얕은 복사는 안쪽 dict를 공유합니다.”
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
print((COURSE / 'exercises/messages.py').read_text())

In [ ]:
# 구분: 수업 핵심 셀
from tests.test_ollama import check_messages
check_messages('exercise')
print('learner_verification=PASS')

> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다. 수업 중에는 시간이 남을 때 읽습니다.

### 참고 정답

자습에서는 필요할 때 정답과 해설을 열어 비교하고 자신의 코드로 재검증합니다. 아래 실행은 자신의 TODO를 변경하지 않습니다.

In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE / 'solutions/messages.py').read_text())

### 정답 해설 — 입력부터 결과까지

**원본 이력 보존:** previous가 있으면 deepcopy로 내부 사전까지 복사하고, 없으면 빈 목록에서 시작합니다. 그 뒤 user 질문 한 건을 끝에 추가해 새 messages를 반환합니다. 질문을 전송하기 전에 기존 대화 이력을 직접 변경하지 않는 것이 핵심입니다.

기존 메시지가 2개라면 반환 목록은 3개, 원래 목록은 계속 2개입니다. 얕은 복사만 하면 안쪽 사전을 바꿀 때 원본에도 반영될 수 있습니다. 서버 요청 성공 뒤 답변을 이력에 확정하는 것은 호출 측의 별도 책임입니다.

**확인 질문:** API 요청이 실패했는데 원래 이력에 질문부터 추가해 두면 재시도 때 어떤 중복이 생길 수 있나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/messages.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
import copy

def build_messages(question, previous=None):
    # 리스트뿐 아니라 내부 dict도 복사하여 호출자의 대화 이력을 보존합니다.
    result=copy.deepcopy(previous) if previous is not None else []
    # 새 질문을 마지막에 추가해 대화의 시간 순서를 유지합니다.
    result.append({'role':'user','content':question})
    return result
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 핵심 셀
check_messages('solution')
print('solution_verification=PASS')

> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다. 수업 중에는 시간이 남을 때 읽습니다.

### 복사해야 하는 이유: alias와 부작용

Python 리스트와 dict는 객체입니다. `messages = previous`는 내용을 복사하는 것이 아니라 같은 리스트를 가리키는 새 이름을 만듭니다. 여기에 `append`하면 호출자가 보관하던 `previous`도 바뀝니다. 얕은 복사 `previous.copy()`는 바깥 리스트만 새로 만들고 안쪽 dict는 공유합니다.

이번 TODO는 `copy.deepcopy`로 바깥 리스트와 내부 dict까지 복사합니다. 함수가 입력을 몰래 바꾸지 않으면 같은 이력으로 다시 시도하거나 테스트하기 쉽습니다. 작은 메시지 목록에서는 이 명확성이 복사 비용보다 중요합니다.

반환 shape는 텐서가 아니라 다음 구조의 Python 자료입니다.

```text
list[
  dict(role=str, content=str),
  dict(role=str, content=str),
  ...
]
```

`role` 철자와 메시지 순서도 모델 입력의 일부입니다. user 질문과 assistant 답의 순서를 뒤집지 않습니다.


In [ ]:
# 구분: 수업 핵심 셀
previous=[{'role':'user','content':'이 대화에서 설비 코드는 NEPES-42입니다. 확인했다고만 답해줘.'}, {'role':'assistant','content':first['message']['content']}]
messages=load_op('messages').build_messages('앞서 말한 설비 코드는 무엇인가요?',previous)
with_history=client.chat(messages,num_predict=64)
print('이전 목록 길이:',len(previous),'API 목록 길이:',len(messages))
print('전송한 이력:',messages)
print('이력 포함 답:',with_history['message']['content'])

> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 3. 이력 관리 클래스

**turn(턴)** 은 여기서 user 질문 하나와 그에 대한 assistant 답 하나의 쌍입니다. 제공되는 `Conversation`은 성공한 user/assistant 쌍만 저장합니다. 요청 전에 user 메시지를 영구 이력에 먼저 추가했다가 서버가 실패하면 답 없는 반쪽 턴이 남을 수 있으므로, 후보 messages를 별도로 만들고 성공한 뒤에만 상태를 갱신합니다. 데이터베이스 처리에 비유하면 작은 transaction처럼 **성공 시 commit, 실패 시 기존 상태 유지** 원칙입니다.

`max_turns=2`이면 최대 2쌍, 즉 user/assistant 메시지 최대 4개를 보관합니다. 한도를 넘으면 가장 오래된 **완전한 턴** 부터 제거해 역할 쌍이 깨지지 않게 합니다. `reset()`은 이 client 객체의 메시지 목록을 비웁니다. 모델 파일을 삭제하거나 서버의 모델을 메모리에서 내리는 동작은 아닙니다.

`max_turns`는 bool이 아닌 양의 정수만 허용합니다. Python에서는 `True`가 산술적으로 1처럼 동작하는 특성이 있지만, `Conversation(client, True)`는 사용자의 설정 실수를 숨길 수 있어 명시적으로 거부합니다.

턴 수 제한은 정확한 token 제한과 다릅니다. `안녕` 한 턴과 긴 보고서 한 턴의 token 수는 크게 다릅니다. 실제 앱은 tokenizer 기반 token 예산, system 메시지 유지, 요약, 최근 이력 선택 등의 정책이 필요합니다. 이번 실습은 짧은 질문과 2048 context 설정으로 상태 관리 원리에 집중합니다.


In [ ]:
# 구분: 수업 핵심 셀
print((COURSE/'common/ollama_client.py').read_text())

In [ ]:
# 구분: 수업 핵심 셀
conversation=Conversation(client,max_turns=2)
conversation.ask('설비 이름은 검사기 A입니다. 기억했다고 짧게 답하세요.')
result=conversation.ask('설비 이름을 말해줘.')
print(result['message']['content'])
assert len(conversation.messages)==4
conversation.reset()
assert conversation.messages==[]
print('이력 초기화 확인')

> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 4. 실패와 이력 경계는 가짜 응답으로 정확히 검사합니다

서버의 자연어 답변을 고정 문자열과 비교하면 모델 표현 변화 때문에 검증이 불안정해집니다.
이력 저장·reset·오래된 턴 제거·실패 시 원본 보존은 결정적인 가짜 응답으로 검사하고, 실제 서버 연결은 별도로 확인합니다.

여기서 **fake client/test double** 은 미리 정한 응답을 즉시 반환하거나 의도적으로 오류를 발생시킵니다. 이를 통해 네트워크 속도와 자연어 표현에 영향받지 않고 다음 질문에 답할 수 있습니다.

- API에 실제로 어떤 messages가 전달됐는가?
- 최대 턴을 넘었을 때 정확히 오래된 두 메시지가 제거됐는가?
- 오류가 발생한 뒤 저장 이력이 이전과 byte 수준에서 같은가?
- `reset()` 뒤 빈 목록인가?

실제 서버 테스트도 필요하지만 목적이 다릅니다. fake 테스트는 **우리 Python 상태 관리 코드** , live 테스트는 **서버 연결·모델 존재·실제 생성 가능 여부** 를 확인합니다.

**검증 범위:** 바로 아래 `tests/test_ollama.py`는 제공 `Conversation`의 reset·턴 제한·실패 시 보존과 참고 options/messages를 검사합니다. 자신의 `exercises/messages.py`는 위 「수업 핵심 TODO — messages」의 `check_messages('exercise')`가 별도로 검사하므로 두 결과를 모두 확인합니다. 가짜 응답 검사는 공용 서버와 모델의 자연어 품질을 검사하지 않습니다.


In [ ]:
# 구분: 수업 핵심 셀
result=subprocess.run([sys.executable,str(COURSE/'tests/test_ollama.py')],capture_output=True,text=True)
print(result.stdout,result.stderr)
result.check_returncode()

> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 5. 터미널 Python 채팅

아래 명령은 **현재 개인 컨테이너의 JupyterLab Terminal** 에서 실행합니다. Python 코드 셀이나 Jetson 호스트 터미널에서 실행하지 않습니다.

```bash
# 이력 없음
python /workspace/04.LLM/scripts/chat_cli.py
# 이력 있음
python /workspace/04.LLM/scripts/chat_cli.py --history
```

`/reset`으로 초기화하고 `/exit`로 종료합니다. 터미널 채팅 로그는 Data/outputs/04.LLM에 새 파일로 저장됩니다.
서버가 없거나 모델이 없으면 오류를 확인하고 로컬 담당자에게 서버·Data 상태를 요청합니다. 패키지 설치·모델 pull을 수강생마다 반복하지 않습니다.

로그 파일은 재현과 디버깅에 유용하지만 질문과 응답 내용이 남습니다. 실제 프로젝트에서는 저장 목적, 접근 권한, 보존 기간, 삭제 방법과 민감정보 마스킹을 정해야 합니다. 이번 교육용 로그에도 비밀번호나 실제 개인정보를 입력하지 마세요.


In [ ]:
# 구분: 수업 핵심 셀
record={'without_history':without_history,'with_history':with_history}
path=OUTPUTS/f'history-comparison-{time.time_ns()}.json'
path.write_text(json.dumps(record,ensure_ascii=False,indent=2))
print(path)

> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 완료·자습

이력 없는/있는 요청의 전송 messages를 비교하고 reset·실패 처리 검증을 통과합니다.
자습(20분): max_turns를 1/2/4로 바꿔 API에 전달되는 목록을 관찰합니다. 답의 품질과 이력 저장 코드의 정확성을 따로 평가합니다.
수업 후 다른 사람이 서버를 사용하지 않을 때만 `ollama stop gemma3:1b`로 모델 메모리를 해제할 수 있습니다. 모델 파일은 Data에 남습니다.

다음 VLM 시간에는 이 클라이언트 구조에 이미지 입력을 추가합니다. 이번 자료에서는 텍스트만 사용합니다.


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다. 수업 중에는 시간이 남을 때 읽습니다.

## 이력 실습에서 자주 생기는 오류와 오해

| 현상 | 원인/오해 | 확인 방법 |
|---|---|---|
| 이전 정보를 기억하지 못함 | 새 user 메시지만 보냄 | 실제 `messages` 전체를 출력해 이전 user/assistant 쌍 확인 |
| 같은 메시지가 반복해서 늘어남 | 입력 리스트를 직접 append하거나 이력을 두 번 합침 | 함수 호출 전후 `previous` 길이와 객체 내용 비교 |
| reset 후에도 모델이 일반 지식을 답함 | reset은 대화 이력만 지우며 사전학습 지식은 남음 | 세션 고유 코드처럼 모델이 원래 알기 어려운 값으로 시험 |
| 서버 오류 뒤 답 없는 user 메시지가 남음 | 성공 전에 내부 상태를 변경함 | 예외 전후 `conversation.messages` 동일성 검사 |
| `max_turns=2`인데 메시지가 4개 | 한 턴은 user+assistant 두 메시지 | 턴과 메시지 수를 구분 |
| 긴 대화에서 앞 정보가 사라짐 | context/턴 제한으로 제거 또는 잘림 | 전달 payload, token 예산, 제거 정책 기록 |
| 자연어 테스트가 가끔 실패함 | 생성 표현을 고정 문자열로 비교 | 상태 관리는 fake client로, 내용 품질은 별도 기준으로 평가 |
| 다른 사용자의 이력이 섞일까 걱정됨 | 서버 공유와 client 상태를 혼동 | 사용자별 `Conversation` 객체·저장 경로를 분리; 전역 공용 목록 금지 |

대화형 프로그램에서 가장 중요한 경계는 **모델의 비결정적인 자연어 생성** 과 **애플리케이션의 결정적인 상태 관리** 를 분리하는 것입니다. 다음 VLA 실습에서도 모델이 제안한 문자열은 검사 가능한 구조와 허용 범위를 통과한 뒤 사용해야 합니다.

<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** messages 검증과 이력 유무·reset·실패 처리 비교

[2일차 슬라이드 45쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s040) · [배포 PDF 45쪽](../슬라이드/2일차.pdf#page=45)로 돌아가 **45–51쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

